Guia ETL:

0. Estandarizar los nombres de las columnas
1. Duplicados
2. Valores nulos o Vacíos
3. Formato de fecha
4. Valores no válidos
5. Estandarización de texto: 
6. Verificación lógica
7. Validaciones adicionales por aseguradora

Opcional: Agregar una columna flag de error

Columnas Estandarizadas:

seguro
fecha_importe
tipo_producto
numero_poliza
prima
prc_comision
monto_comision
canal_venta
estado_pago
productor

In [381]:
import pandas as pd
import openpyxl
import sys
import os

In [382]:
# Añadir la ruta de la carpeta 'functions' al sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'functions')))

In [383]:
workbook = pd.ExcelFile('../data/Comisiones.xlsx')

In [421]:
df_zurich = workbook.parse(0)
df_zurich.head(1)

,Seguro,FechaReporte,Producto,NoPóliza,Prima,Comisión (%),MontoComisión,EstadoPago,Vendedor
0,Zurich,31/10/2024,Moto,NaN,252864.09,5.0,Pendiente,Isabella Moyano Quiroga,NaN


In [385]:
from renombrar_columnas import renombrar_cols

In [386]:
zurich_renombrar_cols = {'Seguro': 'seguro', 'FechaReporte':'fecha_reporte', 
                        'Producto':'producto', 'NoPóliza':'numero_poliza', 
                        'Prima':'prima', 'Comisión (%)':'prc_comision', 
                        'MontoComisión': 'estado_pago','EstadoPago':'canal_venta'}

df_zurich = renombrar_cols(df_zurich, zurich_renombrar_cols)

In [387]:
from duplicados import duplicados_f

In [388]:
df_zurich = duplicados_f(df_zurich)

In [389]:
df_zurich.isnull().sum()

seguro              0
fecha_reporte       0
producto            0
numero_poliza     420
prima               0
prc_comision        0
estado_pago       379
canal_venta         0
Vendedor         4000
dtype: int64

In [390]:
from nulos import tratar_nulos
df_zurich = tratar_nulos(df_zurich, {'numero_poliza': 'Sin Especificar'})

In [392]:
df_zurich.drop('Vendedor', axis=1, inplace=True)

In [394]:
df_zurich = tratar_nulos(df_zurich,{'estado_pago': 'Sin Especificar'})

In [395]:
df_zurich['estado_pago'].unique()

array(['Pendiente', 'Pagado', 'Sin Especificar'], dtype=object)

In [422]:
df_zurich.head(1)

,Seguro,FechaReporte,Producto,NoPóliza,Prima,Comisión (%),MontoComisión,EstadoPago,Vendedor
0,Zurich,31/10/2024,Moto,NaN,252864.09,5.0,Pendiente,Isabella Moyano Quiroga,NaN


In [397]:
df_zurich['monto_comision'] = round(df_zurich['prima'] * (df_zurich['prc_comision'] / 100), 2)

In [423]:
df_zurich.head(1)

,Seguro,FechaReporte,Producto,NoPóliza,Prima,Comisión (%),MontoComisión,EstadoPago,Vendedor
0,Zurich,31/10/2024,Moto,NaN,252864.09,5.0,Pendiente,Isabella Moyano Quiroga,NaN


In [399]:
from mover_columna import mover_columna

In [400]:
df_zurich = mover_columna(df_zurich, 'monto_comision', 6)

In [401]:
from fecha import fecha_formateada

In [402]:
df_zurich = fecha_formateada(df_zurich, 'fecha_reporte')

In [403]:
# Se borraron los valores que no cumplan el requisito, mejorar la lógica.
df_zurich = df_zurich[(df_zurich['prima'] >= 10000) & (df_zurich['prima'] <= 500000)]

In [404]:
# Se borraron los valores que no cumplan el requisito, mejorar la lógica.
df_zurich = df_zurich[(df_zurich['prc_comision'] >= 5) & (df_zurich['prc_comision'] <= 12.5)]

In [405]:
# Se borraron los valores que no cumplan el requisito, mejorar la lógica.
df_zurich = df_zurich[df_zurich['monto_comision'] <= 62500]

In [406]:
from estandarizar_txt import estandarizar_texto

In [407]:
df_zurich = estandarizar_texto(df_zurich, columnas=['seguro', 'producto', 'estado_pago', 'canal_venta', 'numero_poliza'])

In [408]:
df_zurich.to_excel('../data_clean/zurich_clean.xlsx', engine='openpyxl')

In [409]:
df_sancor = workbook.parse(1)

In [424]:
df_sancor.head(1)

,seguro,fecha_reporte,producto,numero_poliza,prima,prc_comision,monto_comision,estado_pago,canal_venta,productor
0,sancor,20/02/2025,monopatín,sp-48206,458022.14,5,22901.11,pendiente,joaquin diaz,luciana soria ramirez


Columnas Estandarizadas:

seguro
fecha_importe
tipo_producto
numero_poliza
prima
prc_comision
monto_comision
canal_venta
estado_pago

In [411]:
df_sancor.head(3)

,empresa,FechaPago,TipoSeguro,N° Póliza,Prima ($),Comisión (%),Comision ($),PagoEstado,AgenteComercial,Productor
0,Sancor,20/02/2025,Monopatín,sp-48206,458022.14,5,22901.11,Pendiente,Joaquin Diaz,Luciana Soria Ramirez
1,Sancor,29/12/2024,Moto,jY-44968,188986.34,10,18898.63,Pagado,Pedro Gonzalez,Pedro Rodriguez Peralta
2,Sancor,2025-03-25,Auto,El-47694,444551.09,5,22227.55,Pendiente,Sol Rodriguez,Pilar Peralta


In [412]:
sancor_renombrar_cols = {'empresa': 'seguro', 'FechaPago':'fecha_reporte', 
                        'TipoSeguro':'producto', 'N° Póliza':'numero_poliza', 
                        'Prima ($)':'prima', 'Comisión (%)':'prc_comision', 
                        'Comision ($)': 'monto_comision','PagoEstado':'estado_pago', 'AgenteComercial': 'canal_venta', 'Productor': 'productor'}

In [413]:
df_sancor = renombrar_cols(df_sancor, sancor_renombrar_cols)

In [414]:
df_sancor = duplicados_f(df_sancor)

In [415]:
df_sancor.isna().sum()

seguro              0
fecha_reporte       0
producto            0
numero_poliza       0
prima               0
prc_comision      132
monto_comision      0
estado_pago         0
canal_venta         0
productor           0
dtype: int64

In [416]:
df_zurich['prc_comision'] = df_zurich['prc_comision'].fillna(df_zurich['prc_comision'].median()) 

In [426]:
df_sancor.head(1)

,seguro,fecha_reporte,producto,numero_poliza,prima,prc_comision,monto_comision,estado_pago,canal_venta,productor
0,sancor,20/02/2025,monopatín,sp-48206,458022.14,5,22901.11,pendiente,joaquin diaz,luciana soria ramirez


In [419]:
df_sancor = estandarizar_texto(df_sancor, columnas=['seguro', 'producto', 'estado_pago', 'canal_venta', 'numero_poliza', 'productor'])

In [420]:
df_sancor.to_excel('../data_clean/sancor_clean.xlsx', engine='openpyxl')